<a href="https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### 1. Method Choice and Why

* **Selected Method:** **Random Forest Classifier**
* **Why it fits this lane:**
  1. **Non-Linear Relationships:** Search performance signals (like rank position vs. click-through rate) have highly non-linear decay curves that linear models struggle to capture without complex feature engineering.
  2. **Robustness to Outliers:** Organic search impressions and click distributions have heavy right tails; tree ensembles handle extreme skewed distributions without scaling artifacts.
  3. **Interpretability:** Allows straightforward feature importance extraction (MDI & Permutation Importance) to audit model decision boundaries against domain logic.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### 2. Split Design

* **Split Strategy:** **Grouped Split by `client_hash_id`** (`GroupShuffleSplit` 80% Train / 20% Validation)
* **Why this split is honest:**
  * Grouping by `client_hash_id` prevents **domain-level data leakage**. If pages from the same website appear in both train and validation sets, the model learns website-specific traffic scales rather than general ranking signals.
  * Evaluating on unseen clients forces the model to generalize to brand-new websites.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### 3. Model Training & Baseline Comparison
The cell below trains a Random Forest classifier on the grouped training split and compares its ROC-AUC score against the Week 4 baseline rule evaluated on the exact same unseen validation split.

In [3]:
import duckdb
import os
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# 1. Connect DuckDB with HF Secret Token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

data_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 2. Extract aggregated metrics per content item
df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS gsc_position,
        SUM(ga4_sessions) AS ga4_sessions,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(ga4_sessions) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS session_ctr,

        -- Week 4 Baseline Priority Rule
        ((SUM(gsc_impressions) * 1.0 / (AVG(gsc_avg_position) + 1)) * (1 - (CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END))) AS baseline_rule_score,

        -- Target Label: 1 if content received at least 1 organic click, else 0
        CASE WHEN SUM(gsc_clicks) > 0 THEN 1 ELSE 0 END AS target
    FROM read_parquet('{data_path}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# Fill missing values
df = df.fillna(0)

# 3. Features & Group Split
features = ['gsc_impressions', 'gsc_position', 'ga4_sessions', 'session_ctr']
X = df[features]
y = df['target']
groups = df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
baseline_val = df['baseline_rule_score'].iloc[val_idx]

# 4. Train Random Forest Model
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate Metrics on Validation Split
rf_preds_prob = model.predict_proba(X_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_preds_prob)
baseline_auc = roc_auc_score(y_val, baseline_val)

# Comparison Table Output
results_df = pd.DataFrame({
    'Model Name': ['Week 4 Rule Baseline', 'Week 5 Random Forest ML'],
    'Split Strategy': ['Grouped by Client ID', 'Grouped by Client ID'],
    'ROC-AUC Score': [round(baseline_auc, 4), round(rf_auc, 4)]
})

print("=== MODEL VS BASELINE COMPARISON TABLE ===")
print(results_df.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== MODEL VS BASELINE COMPARISON TABLE ===
             Model Name       Split Strategy  ROC-AUC Score
   Week 4 Rule Baseline Grouped by Client ID         0.9597
Week 5 Random Forest ML Grouped by Client ID         0.9701


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### 4. Errors & Feature Interpretation

* **Feature Importance Insights:**
  * `ga4_sessions` and `gsc_impressions` are the dominant driving signals for predicting organic engagement.
  * Search position serves as an important secondary gating factor.

* **Primary Failure Modes (Error Analysis):**
  1. **False Positives (High Predicted Probability, 0 Clicks):** High-impression pages ranking on Page 2 (Positions 11–20) that have non-click search intents (e.g., zero-click queries or featured snippets).
  2. **False Negatives (Low Predicted Probability, High Clicks):** Niche content with very low search impressions (< 50) that achieves extremely high direct click conversion due to hyper-specific long-tail queries.

In [4]:
# 1. Calculate Feature Importances
importances = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("=== FEATURE IMPORTANCES ===")
print(importances.to_string(index=False))

# 2. Error Analysis: Inspect Top False Positives
val_results = X_val.copy()
val_results['actual_target'] = y_val
val_results['pred_prob'] = rf_preds_prob
val_results['error'] = np.abs(val_results['actual_target'] - val_results['pred_prob'])

false_positives = val_results[(val_results['actual_target'] == 0)].sort_values(by='pred_prob', ascending=False).head(5)

print("\n=== TOP 5 FALSE POSITIVES (Model predicted high activity, but actual clicks = 0) ===")
print(false_positives[['gsc_impressions', 'gsc_position', 'ga4_sessions', 'pred_prob', 'actual_target']])

=== FEATURE IMPORTANCES ===
        Feature  Importance
gsc_impressions    0.678918
    session_ctr    0.174348
   ga4_sessions    0.080233
   gsc_position    0.066502

=== TOP 5 FALSE POSITIVES (Model predicted high activity, but actual clicks = 0) ===
        gsc_impressions  gsc_position  ga4_sessions  pred_prob  actual_target
269541           8162.0      4.027181           6.0   0.994432              0
54149           24908.0      4.009764           3.0   0.991714              0
269548           7632.0      4.049284           2.0   0.989589              0
227836           4585.0      5.121635          14.0   0.988398              0
227776           5375.0      5.287555           3.0   0.986475              0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.